# Phase 5.2b: EEGNet with Max-Norm constraints & Lower LR

**Phase 5.2 diagnostics:**
- Val_acc peaked at 0.567 (epoch 3) and then declined steadily.
- Val_loss climbed from 0.69 to ~0.80 with wild oscillations.
- Classic runaway overfitting starting around epoch 4-5.

Two Fixes:
1. **Max-Norm Constraints:** applied after every optimizer.step().

    Per the original EEGNet paper (Lawhern et al. 2018):
    - depthwise_conv weights: L2-norm per filter clipped to 1.0
    - classifier weights:    L2-norm per output row clipped to 0.25

    This is a hard-edged regularizer that prevents any single filter from dominating. PyTorch has no built-in max_norm.
 
2. **Lower LR:** 5e-4 instead of 1e-3. Slower, more careful updates.

We write a custom training loop here (instead of reusing phase5_utils.train_model). So that the new step, `model.apply_max_norm_constraints()` is visible and explicit.

In [1]:

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
 
from phase5_utils import (
    load_phase2_data, make_subject_split, make_loaders,
    evaluate_model, count_parameters,
    PHASE5_DIR, SEED,
)
 
torch.manual_seed(SEED)
np.random.seed(SEED)

# EEGNet with Max-Norm Support

EEGNet-8,2 with a method to apply max-norm weight constraints. Architecture identical to Phase 5.2. The only addition is `apply_max_norm_constraints()`, which must be called after every optimizer.step() during training.

In [2]:

class EEGNetMaxNorm(nn.Module):
 
    def __init__(
        self,
        n_classes=2,
        n_channels=32,
        n_samples=640,
        F1=8, D=2, F2=None,
        kernel_length=64,
        dropout=0.5,
    ):
        super().__init__()
        if F2 is None:
            F2 = F1 * D

        # Block 1a: temporal conv
        self.conv_temporal = nn.Conv2d(
            1, F1, kernel_size=(1, kernel_length),
            padding=(0, kernel_length // 2), bias=False,
        )
        self.bn1 = nn.BatchNorm2d(F1)

        # Block 1b: depthwise spatial conv (max-norm 1.0 will be applied here)
        self.depthwise_conv = nn.Conv2d(
            F1, F1 * D, kernel_size=(n_channels, 1),
            groups=F1, bias=False,
        )
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.drop1 = nn.Dropout(dropout)
 
        # Block 2: separable conv
        self.separable_depthwise = nn.Conv2d(
            F1 * D, F1 * D, kernel_size=(1, 16),
            padding=(0, 8), groups=F1 * D, bias=False,
        )
        self.separable_pointwise = nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.drop2 = nn.Dropout(dropout)
 
        # Classifier (max-norm 0.25 will be applied here)
        self._n_features = self._compute_feature_size(n_channels, n_samples)
        self.classifier = nn.Linear(self._n_features, n_classes)

    def _compute_feature_size(self, n_channels, n_samples):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            x = self.conv_temporal(dummy); x = self.bn1(x)
            x = self.depthwise_conv(x);    x = self.bn2(x)
            x = F.elu(x);                  x = self.pool1(x)
            x = self.separable_depthwise(x)
            x = self.separable_pointwise(x); x = self.bn3(x)
            x = F.elu(x);                  x = self.pool2(x)
            return int(x.flatten(1).shape[1])

    def forward(self, x):
        x = self.conv_temporal(x); x = self.bn1(x)
        x = self.depthwise_conv(x); x = self.bn2(x); x = F.elu(x)
        x = self.pool1(x); x = self.drop1(x)
        x = self.separable_depthwise(x)
        x = self.separable_pointwise(x); x = self.bn3(x); x = F.elu(x)
        x = self.pool2(x); x = self.drop2(x)
        x = x.flatten(1)
        x = self.classifier(x)
        return x
    
    """
    Max-Norm Constraints
    -----------------------------------------
    
    Clip the L2 norm of certain weights, per the original EEGNet paper. Call after each optimizer.step().
 
    For Conv2d depthwise_conv with weight shape (F1*D, 1, C, 1):
    - We compute the L2 norm of each output filter (dim 0) across all of its weights (dims 1, 2, 3).
    - If a filter's norm > dw_max, we scale its weights down so the norm equals dw_max. Filters under the cap are untouched.

    For Linear classifier with weight shape (n_classes, n_features):
    - Same idea applied per output row (dim 0), with cap fc_max.

    Why a scale factor instead of a hard clip? We want to preserve the Direction of the weight vector 
    and only reduce its magnitude (clipping would distort the learned pattern.)
    """

    def apply_max_norm_constraints(self, dw_max=1.0, fc_max=0.25):
            
        with torch.no_grad():
            # depthwise spatial conv ----------
            w = self.depthwise_conv.weight                  # (F1*D, 1, 32, 1)
            # norm of each output filter, keepdim so we can broadcast
            norms = w.norm(p=2, dim=(1, 2, 3), keepdim=True)
            # Scale factor: 1 if under cap, else dw_max/norm
            scale = (dw_max / (norms + 1e-8)).clamp(max=1.0)
            w.mul_(scale)

            # classifier linear ----------
            w = self.classifier.weight                       # (n_classes, n_features)
            norms = w.norm(p=2, dim=1, keepdim=True)
            scale = (fc_max / (norms + 1e-8)).clamp(max=1.0)
            w.mul_(scale)

#  Custom Training Loop

Same training loop as phase5_utils.train_model, with One addition. After each optimizer.step() we call model.apply_max_norm_constraints().

In [3]:

def train_eegnet_with_maxnorm(
    model, train_loader, val_loader,
    n_epochs=100, lr=5e-4, weight_decay=1e-4, patience=20,
    device="cpu", criterion=None, monitor="val_acc", verbose=True,
):
    
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
 
    if monitor == "val_loss":
        best_metric, better = float("inf"), lambda a, b: a < b
    elif monitor == "val_acc":
        best_metric, better = -float("inf"), lambda a, b: a > b
    else:
        raise ValueError(f"Bad monitor: {monitor!r}")
 
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
 
    for epoch in range(1, n_epochs + 1):
        # Training --------------------------------
        model.train()
        train_loss_sum, train_n = 0.0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            model.apply_max_norm_constraints()    # ← The New Line
            train_loss_sum += loss.item() * X_batch.size(0)
            train_n += X_batch.size(0)
        train_loss = train_loss_sum / train_n
 
        # Validation --------------------------------
        model.eval()
        val_loss_sum, val_correct, val_n = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss_sum += loss.item() * X_batch.size(0)
                val_correct  += (logits.argmax(dim=1) == y_batch).sum().item()
                val_n        += X_batch.size(0)
        val_loss, val_acc = val_loss_sum / val_n, val_correct / val_n
 
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
 
        if verbose:
            mark_loss = " ←" if monitor == "val_loss" else ""
            mark_acc  = " ←" if monitor == "val_acc"  else ""
            print(f"  Epoch {epoch:3d} | "
                  f"train_loss {train_loss:.4f} | "
                  f"val_loss {val_loss:.4f}{mark_loss} | "
                  f"val_acc {val_acc:.4f}{mark_acc}")
 
        current = val_loss if monitor == "val_loss" else val_acc
        if better(current, best_metric):
            best_metric = current
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} "f"(no improvement in {monitor} for {patience} epochs).")
                break
 
    return best_state, history

# Main

In [4]:

print("PHASE 5.2b — EEGNet-8,2 with max-norm constraints (lr=5e-4)")
print("=" * 70)
 
 
# 1. Load data + split --------------------------------
print("\n[1/5] Loading data and recreating Phase 5.0 split...")
X, y_binary, subjects = load_phase2_data(verbose=False)
train_idx, val_idx, test_idx, split_info = make_subject_split(subjects, n_train=32, n_val=4, n_test=4, seed=SEED)

train_loader, val_loader, test_loader = make_loaders(X, y_binary, train_idx, val_idx, test_idx, batch_size=64)

print(f"  Train: {train_idx.sum()}  Val: {val_idx.sum()}  Test: {test_idx.sum()}")
 
 
# 2. Class weights --------------------------------
print("\n[2/5] Computing class weights from TRAIN set...")
y_train = y_binary[train_idx]
class_counts = np.bincount(y_train, minlength=2)
class_weights_np = class_counts.sum() / (2 * class_counts)
class_weights = torch.FloatTensor(class_weights_np)
print(f"  Class weights: Relaxed={class_weights_np[0]:.3f}, Stress={class_weights_np[1]:.3f}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
 
 
# 3. Build EEGNetMaxNorm --------------------------------
print("\n[3/5] Building EEGNet-8,2 with max-norm...")
device = "cpu"
model = EEGNetMaxNorm(
    n_classes=2, n_channels=32, n_samples=640,
    F1=8, D=2, F2=16, kernel_length=64, dropout=0.5,
).to(device)
n_params = count_parameters(model)
print(f"  Total trainable parameters: {n_params:,}")
print(f"  (Same arch as Phase 5.2 — max-norm is a training-time regularizer,")
print(f"   not extra parameters.)")
 
# Apply max-norm once before training to make sure init is in-bounds
model.apply_max_norm_constraints()
 
with torch.no_grad():
    dummy = torch.zeros(2, 1, 32, 640)
    out = model(dummy)
    print(f"  Forward-pass sanity: input {tuple(dummy.shape)} → output {tuple(out.shape)}")
 
 
# 4. Train --------------------------------
print("\n[4/5] Training with max-norm constraints (lr=5e-4, monitor=val_acc)...")
start = time.time()
best_state, history = train_eegnet_with_maxnorm(
    model, train_loader, val_loader,
    n_epochs=100, lr=5e-4, weight_decay=1e-4,
    patience=20,
    device=device, criterion=criterion, monitor="val_acc", verbose=True,
)
elapsed = time.time() - start
print(f"  Training time: {elapsed:.1f} seconds ({elapsed/60:.1f} min)")
 
model.load_state_dict(best_state)
 
 
# 5. Evaluate --------------------------------
print("\n[5/5] Evaluating on test set...")
results = evaluate_model(model, test_loader, device=device)
 
n0_test = int((y_binary[test_idx] == 0).sum())
n1_test = int((y_binary[test_idx] == 1).sum())
majority_baseline = max(n0_test, n1_test) / (n0_test + n1_test)
pred_counts = np.bincount(results["preds"], minlength=2)
pred_rate_stress = pred_counts[1] / pred_counts.sum()
 
print(f"  Test accuracy : {results['accuracy']:.4f}")
print(f"  Test F1 (pos) : {results['f1']:.4f}")
print(f"  Test F1 macro : {results['f1_macro']:.4f}")
print(f"  Confusion matrix:")
print(f"    {results['confusion_matrix']}")
print(f"  ---- Diagnostic checks ----")
print(f"  Model's prediction rate of 'Stress': {pred_rate_stress:.3f}  (true 0.396)")
print(f"  Best val_acc reached during training: {max(history['val_acc']):.4f}")
print(f"  ---- Reference points ----")
print(f"  Majority-class baseline on test  : {majority_baseline:.4f}")
print(f"  Phase 4B best (binary RF, LOSO)  : 0.5870")
print(f"  Phase 5.1  (no class weights)    : 0.4833")
print(f"  Phase 5.1b (smaller + weights)   : 0.5500")
print(f"  Phase 5.2  (EEGNet, no max-norm) : 0.5167")

PHASE 5.2b — EEGNet-8,2 with max-norm constraints (lr=5e-4)

[1/5] Loading data and recreating Phase 5.0 split...
  Train: 1920  Val: 240  Test: 240

[2/5] Computing class weights from TRAIN set...
  Class weights: Relaxed=1.136, Stress=0.893

[3/5] Building EEGNet-8,2 with max-norm...
  Total trainable parameters: 2,258
  (Same arch as Phase 5.2 — max-norm is a training-time regularizer,
   not extra parameters.)
  Forward-pass sanity: input (2, 1, 32, 640) → output (2, 2)

[4/5] Training with max-norm constraints (lr=5e-4, monitor=val_acc)...
  Epoch   1 | train_loss 0.6943 | val_loss 0.6908 | val_acc 0.4792 ←
  Epoch   2 | train_loss 0.6950 | val_loss 0.6885 | val_acc 0.4792 ←
  Epoch   3 | train_loss 0.6896 | val_loss 0.6885 | val_acc 0.5542 ←
  Epoch   4 | train_loss 0.6908 | val_loss 0.6861 | val_acc 0.5417 ←
  Epoch   5 | train_loss 0.6854 | val_loss 0.6903 | val_acc 0.5375 ←
  Epoch   6 | train_loss 0.6841 | val_loss 0.6906 | val_acc 0.5292 ←
  Epoch   7 | train_loss 0.6791 | v

# Save Results and Plots

In [5]:

out_dir = os.path.join(PHASE5_DIR, "phase5_2b_eegnet_maxnorm")
os.makedirs(out_dir, exist_ok=True)
 
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs, history["train_loss"], label="Train", color="#5DA5DA", marker="o", markersize=3)
axes[0].plot(epochs, history["val_loss"],   label="Val",   color="#F15854", marker="o", markersize=3)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Weighted CE loss")
axes[0].set_title("Loss curves (class-weighted)"); axes[0].legend(); axes[0].grid(alpha=0.3)
 
axes[1].plot(epochs, history["val_acc"], color="#60BD68", marker="o", markersize=3)
axes[1].axhline(majority_baseline, color="gray", linestyle="--", label=f"Majority baseline ({majority_baseline:.3f})")
axes[1].axhline(0.5870, color="purple", linestyle=":", label="Phase 4B RF (0.587)")
axes[1].axhline(0.6125, color="orange", linestyle=":", label="Phase 5.1b best val_acc (0.6125)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Validation accuracy (monitored metric)")
axes[1].legend(loc="lower right"); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1)
 
plt.suptitle(f"Phase 5.2b — EEGNet-8,2 + max-norm ({n_params:,} params)")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
plt.close()
 
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(results["confusion_matrix"], annot=True, fmt="d", cmap="Blues", xticklabels=["Relaxed", "Stress"], yticklabels=["Relaxed", "Stress"], ax=ax,)

ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Phase 5.2b (EEGNet + max-norm) test confusion matrix\n"f"acc={results['accuracy']:.3f}, F1-macro={results['f1_macro']:.3f}")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
plt.close()
 
# JSON summary
results_summary = {
    "model": "EEGNet-8,2 + max-norm",
    "n_params": int(n_params),
    "training_time_sec": float(elapsed),
    "n_epochs_trained": len(history["train_loss"]),
    "test_accuracy": float(results["accuracy"]),
    "test_f1": float(results["f1"]),
    "test_f1_macro": float(results["f1_macro"]),
    "majority_baseline": float(majority_baseline),
    "predicted_stress_rate": float(pred_rate_stress),
    "best_val_acc": float(max(history["val_acc"])),
    "best_val_loss": float(min(history["val_loss"])),
    "confusion_matrix": results["confusion_matrix"].tolist(),
    "split_subjects": split_info,
    "hyperparameters": {
        "F1": 8, "D": 2, "F2": 16,
        "kernel_length": 64,
        "dropout": 0.5,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "max_epochs": 100,
        "patience": 20,
        "monitor": "val_acc",
        "class_weighted_loss": True,
        "max_norm_depthwise": 1.0,
        "max_norm_classifier": 0.25,
    },
    "comparison": {
        "phase4b_rf_loso":      0.5870,
        "phase5_1_simple":      0.4833,
        "phase5_1b_simple_v2":  0.5500,
        "phase5_2_eegnet":      0.5167,
        "phase5_2b_eegnet_maxnorm": float(results["accuracy"]),
    },
}
with open(os.path.join(out_dir, "results.json"), "w") as f:
    json.dump(results_summary, f, indent=2)
 
torch.save(best_state, os.path.join(out_dir, "eegnet_maxnorm_best.pt"))
 
print(f"\nSaved outputs to: {out_dir}")
print("=" * 70)
print("Phase 5.2b complete.")
print("=" * 70)


Saved outputs to: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\phase5_2b_eegnet_maxnorm
Phase 5.2b complete.
